In [173]:
import sys
import os
import json
import pandas as pd
import numpy as np
import pandas as pd
from hmmlearn import hmm

# Step 2: Get the absolute path to the src folder and add it to the system path
# Assuming you are running the notebook from the project root directory
src_path = os.path.abspath(os.path.join('..', 'src'))
sys.path.append(src_path)

# Step 3: Import the function from the data_preprocessing module
from data_preprocessing import save_assistments_BKT

In [2]:
save_assistments_BKT()

with open('../data/preprocessed/assistments_skill_dict.json', 'r') as json_file:
    skill_dict = json.load(json_file)

Skill dictionary saved to ../data/preprocessed/assistments_skill_dict.json.


In [93]:
class BKT:
    def __init__(self, skill_id, user_answers, n_states=2, initial_probs=None, trans_probs=None, emit_probs=None):
        """
        Initializes a BKT model for a specific skill using HMM.

        Parameters:
        - skill_id: The ID of the skill for which the BKT model is created.
        - n_states: Number of hidden states (default: 2, representing knowledge and lack of knowledge).
        """
        # Initialize parameters
        self.skill_id = skill_id
        self.user_answers = user_answers
        self.n_users = len(user_answers)
        self.n_states = n_states
        self.initial_probs = initial_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs
        self.model = None





        # HMM parameters
     #   self.model = 

        # Initialize parameters


    def set_initial_parameters(self):

        correct_first_answers = 0

        answer_pairs = {('No', 'No'): 0,
               ('No', 'Yes'): 0,
               ('Yes', 'No'): 0,
               ('Yes', 'Yes'): 0}
        
        for answers in self.user_answers:
            if answers[0][0] == 1:
                correct_first_answers +=1

            previous_answer = 0 # Assume starting state
            for answer in answers:
                answer = answer[0]
                if (previous_answer == 0) & (answer == 0):
                    answer_pairs[('No', 'No')] += 1
                if (previous_answer == 0) & (answer == 1):
                    answer_pairs[('No', 'Yes')] += 1
                if (previous_answer == 1) & (answer == 0):
                    answer_pairs[('Yes', 'No')] += 1
                if (previous_answer == 1) & (answer == 1):
                    answer_pairs[('Yes', 'Yes')] += 1
                previous_answer = answer

        incorrect_first_answers = self.n_users - correct_first_answers

        self.initial_probs = np.array([correct_first_answers / self.n_users,
                                       incorrect_first_answers / self.n_users])
        
        no_yes_counts = answer_pairs[('No', 'Yes')] / 2
        guesses = answer_pairs[('No', 'Yes')] / 2
        no_no_counts = guesses + answer_pairs[('No', 'No')]
        no_counts = answer_pairs[('No', 'Yes')] + answer_pairs[('No', 'No')]
        slips = answer_pairs[('Yes', 'No')]
        yes_yes_counts = answer_pairs[('Yes', 'Yes')]
        yes_counts = answer_pairs[('Yes', 'Yes')] + answer_pairs[('Yes', 'No')]

        self.trans_probs = np.array([[no_no_counts / no_counts,
                                      no_yes_counts / no_counts],
                                      [0, 1]])

        self.emit_probs = np.array([[no_no_counts / no_counts,
                            guesses / no_counts],
                            [slips / yes_counts,
                            yes_yes_counts / yes_counts]])



    def fit(self):
        """
        Fit the BKT model using user answers.
        """
        self.model = hmm.CategoricalHMM(n_components=self.n_states)

        #if (self.initial_probs is None) or (self.trans_probs is None) or (self.emit_probs is None):
         #   self.set_initial_parameters()

        self.model.startprob_ = self.initial_probs
        self.model.transmat_ = self.trans_probs
        self.model.emissionprob_ = self.emit_probs
        
        # Fit the HMM model
        X = np.concatenate([answers for answers in self.user_answers])
        lengths = [len(answers) for answers in self.user_answers]
        self.model.fit(X, lengths)

        # Store learned parameters
        self.initial_probs = self.model.startprob_
        self.trans_probs = self.model.transmat_
        self.emit_probs = self.model.emissionprob_

    def predict(self, user_answers):
        """
        Predict the hidden states for the given user answers.

        Parameters:
        - user_answers: A list of lists containing binary answers (0 or 1) for the skill.
        
        Returns:
        - states: An array of predicted hidden states.
        """
        X = np.concatenate(user_answers).reshape(-1, 1)
        states = self.model.predict(X)
        return states

    def get_params(self):
        """
        Retrieve the learned parameters of the HMM.
        
        Returns:
        - A dictionary containing initial, transition, and emission probabilities.
        """
        return {
            'initial_probs': self.initial_probs,
            'trans_probs': self.trans_probs,
            'emit_probs': self.emit_probs
        }




In [172]:
def viterbi(obs, states, start_p, trans_p, emit_p):
    V = [{}]
    path = {}

    for y in states:
        V[0][y] = start_p[y] * emit_p[y][obs[0]]
        path[y] = [y]

    for t in range(1, len(obs)):
        V.append({})
        newpath = {}

        for y in states:
            (prob, state) = max(
                [(V[t-1][y0] * trans_p[y0][y] * emit_p[y][obs[t]], y0) for y0 in states]
            )
            V[t][y] = prob
            newpath[y] = path[state] + [y]

        path = newpath

    (prob, state) = max([(V[-1][y], y) for y in states])
    return (prob, path[state])

states = ('0', '1')
observations = ('0', '0', '1', '0', '1', '1')
start_probability = {'0': 0.9, '1': 0.1}
transition_probability = {
   '0' : {'0': 0.7, '1': 0.3},
   '1' : {'0': 0, '1': 1},
   }
emission_probability = {
   '0' : {'0': 0.6, '1': 0.4},
   '1' : {'0': 0.2, '1': 0.8},
   }

print(viterbi(observations, states, start_probability, transition_probability, emission_probability))

(0.006967296000000002, ['0', '0', '1', '1', '1', '1'])
